## 构建数据集

In [1]:
import os

eval_ids = set()
with open(os.path.join('data', 'emb_eval'), 'r') as f:
    for l in f:
        id = l.strip().split('\t')[0]
        eval_ids.add(id)
print(f'eval_ids: {len(eval_ids)}')


train_0331_ids = set()
with open(os.path.join('data', 'nid_train_0406.txt'), 'r') as f:
    for l in f:
        id = l.strip().split('\t')[0]
        train_0331_ids.add(id)
print(f'train_0331_ids: {len(train_0331_ids)}')

eval_ids: 385052
train_0331_ids: 274504


In [ ]:
## 构造原始数据集
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import os
files = ['data/emb_eval']
output_dir = 'data/eval'
os.makedirs(output_dir, exist_ok=True)

dim = 1024

schema = pa.schema([
    ('id', pa.string()),
    ('embedding', pa.list_(pa.float64(), dim))
])

parquet_id = 0
parquet_samples = 250000

ids = []
id_set = set()
embeddings = []

def flush_shard(ids, embeddings, parquet_id):
    if not ids:
        return parquet_id
    print(len(ids))
    table = pa.Table.from_arrays([ids, embeddings], schema=schema)
    pq.write_table(table, os.path.join(output_dir, f'shard-{parquet_id:04d}.parquet'))
    return parquet_id + 1

for i, file_name in enumerate(files):
    with open(file_name, 'r') as f:
        for line in f:
            parts = line.strip('[]').split()
            id = str(parts[0])
            if id in id_set or id not in eval_ids:
                continue
            id_set.add(id)
            ids.append(id)
            embeddings.append([float(part.strip('[],')) for part in parts[1:] if part.strip('[],')])
            if len(ids) >= parquet_samples:
                parquet_id = flush_shard(ids, embeddings, parquet_id)
                ids = []
                embeddings = []
flush_shard(ids, embeddings, parquet_id)


250000
135052


2

In [1]:
## 构造对比学习的数据集
import sqlite3
import pyarrow as pa
import pyarrow.parquet as pq
import struct
import os

pair_file = 'data/nid_pairs_test.txt'
emb_file = 'data/emb_cl_test'
output_dir = 'data/cl_test/'
db_path = 'data/emb_index.db'
os.makedirs(output_dir, exist_ok=True)

dim = 1024
parquet_samples = 125000
pack_fmt = f'{dim}d'

# 读取所有pair对里面的 nid
need_nids = set()
with open(pair_file, 'r') as f:
    for line in f:
        id1, id2 = line.strip().split()
        need_nids.add(id1)
        need_nids.add(id2)
print(f'need_ids: {len(need_nids)}')


# 构建sql，方便查询embedding
conn = sqlite3.connect(db_path)
conn.execute('CREATE TABLE IF NOT EXISTS emb (nid TEXT PRIMARY KEY, data BLOB)')
conn.execute('DELETE FROM emb')

batch_buf = []
inserted = 0
with open(emb_file, 'r') as f:
    for line in f:
        parts = line.strip('[]').split()
        nid = str(parts[0])
        if nid not in need_nids:
            continue
        emb = [float(p.strip('[],')) for p in parts[1:] if p.strip('[],')]
        blob = struct.pack(pack_fmt, *emb)
        batch_buf.append((nid, blob))
        if len(batch_buf) >= 10000:
            conn.executemany("INSERT OR IGNORE INTO emb VALUES (?, ?)", batch_buf)
            conn.commit()
            inserted += len(batch_buf)
            batch_buf = []
if batch_buf:
    conn.executemany("INSERT OR IGNORE INTO emb VALUES (?, ?)", batch_buf)
    conn.commit()
    inserted += len(batch_buf)
print(f'Inserted {inserted} embeddings into database.')

# 构建数据集
schema = pa.schema([
    ('id_a', pa.string()),
    ('embedding_a', pa.list_(pa.float64(), dim)),
    ('id_b', pa.string()),
    ('embedding_b', pa.list_(pa.float64(), dim))
])

parquet_id = 0
buf_id_a, buf_emb_a, buf_id_b, buf_emb_b = [], [], [], []
skipped = 0

def flush_shard(parquet_id):
    if not buf_id_a:
        return parquet_id
    table = pa.Table.from_arrays(
        [buf_id_a, buf_emb_a, buf_id_b, buf_emb_b],
        schema=schema
    )
    pq.write_table(table, os.path.join(output_dir, f'pairs-{parquet_id:04d}.parquet'))
    return parquet_id + 1

def lookup(nid):
    row = conn.execute("SELECT data FROM emb WHERE nid=?", (nid,)).fetchone()
    if row is None:
        return None
    return list(struct.unpack(pack_fmt, row[0]))

with open(pair_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        id_a, id_b = parts
        emb_a = lookup(id_a)
        emb_b = lookup(id_b)
        if emb_a is None or emb_b is None:
            skipped += 1
            continue
        buf_id_a.append(id_a)
        buf_emb_a.append(emb_a)
        buf_id_b.append(id_b)
        buf_emb_b.append(emb_b)
        if len(buf_id_a) >= parquet_samples:
            parquet_id = flush_shard(parquet_id)
            buf_id_a, buf_emb_a, buf_id_b, buf_emb_b = [], [], [], []
parquet_id = flush_shard(parquet_id)
conn.close()
os.remove(db_path)
print(f'Done. {parquet_id} shards, skipped {skipped} pairs.')


need_ids: 4272
Inserted 4226 embeddings into database.
Done. 1 shards, skipped 119 pairs.


## 测试模型

In [5]:
import json

from modules.rqvae import RqVae
from modules.rqkmeans import RqKmeans
from modules.evaluate import evaluate
from train_rqkmeans import get_dataset

import torch
from torch.utils.data import DataLoader

dataset = get_dataset('data/eval', seed=42, normalize=True)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=False, num_workers=4)

checkpoint_path = '/root/paddlejob/workspace/Users/chengfeilin/sid/output/rqvae/7.prefixinfonce+prefixcollision/checkpoint_final.pt'
state_dict = torch.load(checkpoint_path, map_location='cpu')

model_config = state_dict['config']
model = RqVae.from_config(model_config)
# model = RqKmeans(**model_config)
model.load_state_dict(state_dict['model'])
device = 'cuda:0'
model.to(device)

metrics, collision_ids = evaluate(model, dataloader, model_config, None)

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

with open('collision_ids.json', 'w') as f:
    json.dump(collision_ids, f, indent=2)

Filter: 100%|██████████| 385052/385052 [00:01<00:00, 251094.25 examples/s]


In [6]:
def get_collision_ids(path):
    with open(path, 'r') as f:
        collision_ids = json.load(f)
    return collision_ids

In [ ]:
## 生成pair对
nid_pairs = []


for key, val in collision_ids['pre_3_layer'].items():
    if len(val) > 1:
        for i in range(len(val)):
            for j in range(i + 1, len(val)):
                nid_pairs.append([val[i], val[j]])

import random

samples = random.sample(nid_pairs, min(len(nid_pairs), 100))
nids = []

with open('nids.txt', 'w') as f:
    for sample in samples:
        f.write(f"{sample[0]}\n{sample[1]}\n")

nid_set = set()
for sample in samples:
    nid_set.add(sample[0])
    nid_set.add(sample[1])

## 检测pair对初始embedding的相似度
filtered_dataset = dataset.filter(lambda x: x['id'] in nid_set)

embedding_dict = {}
for data in filtered_dataset:
    nid = data['id']
    embedding_dict[nid] = data['embedding']
embeddings1 = []
embeddings2 = []
for nid0, nid1 in samples:
    embeddings1.append(embedding_dict[nid0])
    embeddings2.append(embedding_dict[nid1])
embeddings1 = torch.stack(embeddings1)
embeddings2 = torch.stack(embeddings2)
print((embeddings1 * embeddings2).sum(dim=-1))

In [7]:
## 查看 bad case
from collections import defaultdict
cos_threshold = 0.6
bad_cases_dict = defaultdict(list)
layer = 'pre_3_layer'
device = 'cuda:7'

dataset = get_dataset('/root/paddlejob/workspace/Users/chengfeilin/gr/data/train_0406', seed=42, normalize=True)
collision_ids = get_collision_ids('/root/paddlejob/workspace/Users/chengfeilin/gr/output/rqvae/2.simvq+rotation_trick/collision_ids.json')

id2embedding = {}
idset = set()
for sid, ids in collision_ids[layer].items():
    ids = set(ids)
    idset.update(ids)
dataset_filtered = dataset.filter(lambda x: x['id'] in idset)

for data in dataset_filtered:
    id2embedding[data['id']] = data['embedding']

print(len(id2embedding))
for sid, ids in collision_ids[layer].items():
    embeddings = []
    for id in ids:
        emb = id2embedding[id]
        embeddings.append(emb)
    embeddings = torch.stack(embeddings).to(device)

    sim = embeddings @ embeddings.T
    lower_mask = torch.tril(torch.ones_like(sim), diagonal=-1).bool()
    idx1, idx2 = torch.where((sim < cos_threshold) & lower_mask)

    for i1, i2 in zip(idx1.cpu().numpy(), idx2.cpu().numpy()):
        id1, id2 = ids[i1], ids[i2]
        bad_cases_dict[sid].append((id1, id2, sim[i1, i2].item())) 

with open('bad_cases.txt', 'w') as f:
    for sid, cases in bad_cases_dict.items():
        for id1, id2, sim in cases:
            f.write(f'{id1}\n{id2}\n')
with open('bad_cases.json', 'w') as f:
    json.dump(bad_cases_dict, f, indent=4)

Filter: 100%|██████████| 257667/257667 [00:01<00:00, 159632.88 examples/s]


25586


In [18]:
# 对每一层sid进行统计
import numpy as np
for key, layer_dict in collision_ids.items():
    layer_max = 0
    layer_sid_nids_len = []
    sid = 0
    for k, v in layer_dict.items():
        layer_sid_nids_len.append([len(v)])
        if len(v) > layer_max:
            layer_max = len(v)
            sid = k
    statics = np.array(layer_sid_nids_len)
    print(f'max: {layer_max}')
    print(f'mean: {statics.mean()}')
    print(f'std: {statics.std()}')
    with open(f'{key}_{sid}.txt', 'w') as f:
        for p in layer_dict[sid]:
            f.write(f'{p}\n')

max: 3248
mean: 265.1733870967742
std: 324.4190343618412
max: 671
mean: 3.958785597907153
std: 6.734361882452598
max: 82
mean: 2.5882412879559813
std: 2.129855199644337


In [54]:
## 抽取第一层的sid
layer_0_sid_dict = collision_ids['pre_1_layer']
id = '[1000]'
nids = layer_0_sid_dict[id]
print(len(nids))
print(nids)
with open('layer_one_ids.txt', 'w') as f:
    for nid in nids:
        f.write(nid + '\n')

69
['3738136593620595729', '4197786278642325482', '5071581045649406607', '3081082540977029067', '5258786450941445675', '4441725075189470928', '4704148427558998986', '4940672439750406934', '4100721177465614070', '4123201117745097283', '3965727301769828522', '4524950705310540826', '4730721737814534124', '5005959000498047320', '4115265092137629080', '4056405773578993102', '5456999840307931216', '5140564903438908635', '3893670940094864995', '5230387036296923276', '4036154453026560023', '4301652370342402796', '4559591106780537770', '2925252679281201795', '4820567858325037334', '5167200112958909776', '3859247133784101256', '3897623091873857028', '4159394880484787155', '7303694676330911494', '4720014500394481401', '4821797197946341429', '3390897262306084991', '14383938264227744014', '5025482169564234994', '3808308602736851414', '5529036771206888395', '5595602888701186656', '4899834786683536950', '4287904064170003348', '4518955729876204723', '5174212824024548310', '5269728661797941794', '45894

In [ ]:
## 查看原始数据相似度
from train_rqvae import get_dataset
from torch.utils.data import DataLoader
import torch

dataset = get_dataset('data/100/train_0331', 42, normalize=True)

dataloader = DataLoader(dataset, batch_size=1024, shuffle=False)

cnt = 0
device = 'cuda:0'

for batch in dataloader:
    embeddings = batch['embedding']
    embeddings = embeddings.to(device)
    sim = embeddings @ embeddings.T
    sim.tril_(diagonal=-1)
    if cnt == 0:
        print(sim)
    ids, _ = torch.where(sim > 0.81)
    cnt += ids.size(0)
print(2 * cnt / (len(dataset) * 1023))

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.4541, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3358, 0.5116, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.3253, 0.3743, 0.3013,  ..., 0.0000, 0.0000, 0.0000],
        [0.3367, 0.2730, 0.2575,  ..., 0.1797, 0.0000, 0.0000],
        [0.3255, 0.3181, 0.2848,  ..., 0.1871, 0.3593, 0.0000]],
       device='cuda:0')
0.004555582375412504


# 推理阶段

In [ ]:
## 无pair

from modules.rqvae import RqVae
from train_rqvae import get_dataset

import torch
from torch.utils.data import DataLoader

state = torch.load("/root/paddlejob/workspace/Users/chengfeilin/sid/output/rqvae/7.prefixinfonce+prefixcollision/checkpoint_final.pt")
dataset = get_dataset("data/train", seed=42, normalize=True)
dataloader = DataLoader(dataset, batch_size=8196, shuffle=False, num_workers=4)
config = state['config']
model = RqVae.from_config(config)
model.load_state_dict(state['model'])
device = "cuda:0"
model.to(device)

all_nids = []
all_sids = []

for batch in dataloader:
    embs = batch['embedding']
    embs = embs.to(model.device)
    output = model(embs)
    sids = output.sids
    nids = batch['id']
    all_nids.extend(nids)
    all_sids.append(sids)

all_sids = torch.concat(all_sids, dim=0)
all_sids = all_sids.cpu().numpy().tolist()

import polars as pl
df = pl.DataFrame({"nid": all_nids, "sid": all_sids})
df.write_json("output.json")


In [2]:
# 有pair

from modules.rqvae import RqVae
from train_rqvae import get_pair_dataset

import torch
from torch.utils.data import DataLoader

state = torch.load("/root/paddlejob/workspace/Users/chengfeilin/sid/output/rqvae/7.prefixinfonce+prefixcollision/checkpoint_final.pt")
dataset = get_pair_dataset("data/cl_test", seed=42, normalize=True)
dataloader = DataLoader(dataset, batch_size=8196, shuffle=False, num_workers=4)
config = state['config']
model = RqVae.from_config(config)
model.load_state_dict(state['model'])
device = "cuda:0"
model.to(device)

all_id_a = []
all_id_b = []
all_sid_a = []
all_sid_b = []

for batch in dataloader:
    emb_a = batch['embedding_a'].to(device)
    emb_b = batch['embedding_b'].to(device)
    nid_a = batch['id_a']
    nid_b = batch['id_b']
    with torch.no_grad():
        output_a = model(emb_a)
        sid_a = output_a.sids
        output_b = model(emb_b)
        sid_b = output_b.sids
        all_id_a.extend(nid_a)
        all_id_b.extend(nid_b)
        all_sid_a.append(sid_a)
        all_sid_b.append(sid_b)

all_sid_a = torch.cat(all_sid_a).cpu().numpy().tolist()
all_sid_b = torch.cat(all_sid_b).cpu().numpy().tolist()

import polars
df = polars.DataFrame({
    "id_a": all_id_a,
    "id_b": all_id_b,
    "sid_a": all_sid_a,
    "sid_b": all_sid_b
})
df.write_json("cl_pair_test.json")



Generating train split: 4998 examples [00:00, 20905.47 examples/s]
Map: 100%|██████████| 4998/4998 [00:03<00:00, 1300.90 examples/s]
